In [ ]:
def air_density_at(altitude_metres):
    # Constants
    sea_level_pressure = 101325  # Sea level standard atmospheric pressure in Pa
    sea_level_temperature = 288.15  # Sea level standard temperature in K
    temperature_lapse_rate = 0.0065  # Temperature lapse rate in K/m
    gas_constant_air = 287.05  # Specific gas constant for dry air in J/(kg·K)
    gravity = 9.8  # Acceleration due to gravity in m/s^2
    
    # Calculate temperature at the given altitude
    temperature = sea_level_temperature - temperature_lapse_rate * altitude_metres
   
    # Calculate pressure at the given altitude using the barometric formula
    pressure = sea_level_pressure * (temperature / sea_level_temperature) ** (gravity / (temperature_lapse_rate * gas_constant_air))
    
    # Calculate air density using the ideal gas law
    return pressure / (gas_constant_air * temperature)

In [ ]:
def vector_sum(*list_of_vectors):
    sum_x = 0
    sum_y = 0
    for (x, y) in list_of_vectors:
        sum_x += x
        sum_y += y
    return (sum_x, sum_y)  

In [ ]:
from vector_2d import scale_vector, vector_from_magnitude_and_angle, vector_angle_radians, vector_magnitude, component_of_vector_in_direction_radians, normalize_angle

In [ ]:
import math

def air_density(aircraft):
    return air_density_at(aircraft.position[1])  
    
def airspeed(aircraft):
    return vector_magnitude(aircraft.linear_velocity)
    
def propeller_area(aircraft):
    return math.pi * (aircraft.propeller_diameter / 2)**2
    
def propeller_rpm(aircraft):
    return aircraft.engine_rpm / aircraft.propeller_gearing_ratio
        
def propeller_radians_per_second(aircraft):
    return propeller_rpm(aircraft) * 2 * math.pi / 60.0    
    
def advance_ratio(aircraft):
    return  airspeed(aircraft) / ((propeller_rpm(aircraft) / 60) * aircraft.propeller_diameter) 
    
def propeller_torque_load(aircraft):
    return  aircraft.propeller_torque_coefficient(advance_ratio(aircraft)) * air_density(aircraft) * propeller_radians_per_second(aircraft)**2 * aircraft.propeller_diameter**5
    
def engine_torque_load(aircraft):
    return propeller_torque_load(aircraft) / aircraft.propeller_gearing_ratio
    
def engine_torque_generated(aircraft):
    return  aircraft.engine_torque(aircraft.throttle, aircraft.engine_rpm)
    
def net_engine_torque(aircraft):
    return  engine_torque_generated(aircraft) - engine_torque_load(aircraft)
    
def engine_angular_acceleration(aircraft):
    return  net_engine_torque(aircraft) / aircraft.engine_moment_of_inertia
    
def update_engine_rpm(aircraft, delta_time):
    angular_acceleration = engine_angular_acceleration(aircraft)
    change_in_angular_velocity = angular_acceleration * delta_time  
    change_in_rpm = change_in_angular_velocity * 60 / math.pi
    aircraft.engine_rpm += change_in_rpm

    if aircraft.engine_rpm > aircraft.engine_max_rpm:
        aircraft.engine_rpm = aircraft.engine_max_rpm

def engine_thrust_force(aircraft):
    magnitude = aircraft.propeller_thrust_coefficient(advance_ratio(aircraft)) * air_density(aircraft) * propeller_radians_per_second(aircraft)**2 * aircraft.propeller_diameter**4   
    return vector_from_magnitude_and_angle(magnitude, aircraft.pitch_angle_radians)    

def induced_drag_magnitude(aircraft, magnitude_lift, wing_span, wing_area, oswald_efficency_factor):
    if airspeed(aircraft) == 0:
        return 0
    else:
        aspect_ratio = wing_span ** 2 / wing_area 
        return magnitude_lift**2 / (0.5 * air_density(aircraft) * airspeed(aircraft)**2 * wing_area * math.pi * oswald_efficency_factor * aspect_ratio)    

def angle_to_relative_airflow_radians(aircraft) : 
    return vector_angle_radians(aircraft.linear_velocity)

def magnitude_of_lift(aircraft, aerofoil_area, compute_lift_coefficient, angle_of_aerofoil_radians):    
    angle_of_attack_radians = angle_of_aerofoil_radians - angle_to_relative_airflow_radians(aircraft)
    lift_coefficient = compute_lift_coefficient(angle_of_attack_radians)
    return 0.5 * air_density(aircraft) * airspeed(aircraft)**2 * aerofoil_area * lift_coefficient
    
def lift_force(aircraft, magnitude_lift):        
    angle_normal_to_relative_airflow_radians = angle_to_relative_airflow_radians(aircraft) + math.radians(90)
    return vector_from_magnitude_and_angle(magnitude_lift, angle_normal_to_relative_airflow_radians)    
    
def induced_drag_force(aircraft, magnitude_lift, aerofoil_span, aerofoil_area, oswald_efficency_factor) :   
    magnitude_induced_drag = induced_drag_magnitude(aircraft, magnitude_lift, aerofoil_span, aerofoil_area, oswald_efficency_factor)
    return vector_from_magnitude_and_angle(- magnitude_induced_drag, angle_to_relative_airflow_radians(aircraft))

def total_aerodynamic_force(aircraft, aerofoil_area, aerofoil_span, oswald_efficency_factor, compute_lift_coefficient, angle_of_aerofoil_radians):  
    magnitude_lift = magnitude_of_lift(aircraft, aerofoil_area, compute_lift_coefficient, angle_of_aerofoil_radians)
    return vector_sum(lift_force(aircraft, magnitude_lift), induced_drag_force(aircraft, magnitude_lift, aerofoil_span, aerofoil_area, oswald_efficency_factor))    
    
def wing_aerodynamic_force(aircraft):
    angle_of_wings_radians = aircraft.pitch_angle_radians + aircraft.wing_angle_of_incidence_radians
    return total_aerodynamic_force(aircraft, aircraft.wing_area, aircraft.wing_span, aircraft.wing_oswald_efficency_factor, aircraft.wing_lift_coefficient, angle_of_wings_radians)   

def stabalizer_aerodynamic_force(aircraft):
    angle_of_stabilizer_radians = aircraft.pitch_angle_radians + aircraft.stabilizer_angle_radians
    return total_aerodynamic_force(aircraft, aircraft.stabilizer_area, aircraft.stabilizer_span, aircraft.stabilizer_oswald_efficency_factor, aircraft.stabilizer_lift_coefficient, angle_of_stabilizer_radians)

def parasitic_drag_force(aircraft):
    magnitude = 0.5 * air_density(aircraft) * airspeed(aircraft) ** 2 * aircraft.drag_coefficient * aircraft.cross_sectional_area
    return vector_from_magnitude_and_angle(- magnitude, angle_to_relative_airflow_radians(aircraft))   

def weight_force(aircraft):
    gravity = 9.8  # Gravitational acceleration (m/s^2)
    return (0, -aircraft.mass * gravity)

def is_on_ground(aircraft):
    return aircraft.position[1] <= 0

def rolling_resistance_force(aircraft):
    if is_on_ground(aircraft) :
        rolling_resistance_coefficient = 0.04
        weight = weight_force(aircraft)[1]
        if aircraft.linear_velocity[0] > 0 :
            return (weight * rolling_resistance_coefficient, 0)
        elif aircraft.linear_velocity[0] < 0 :
            return (-weight * rolling_resistance_coefficient, 0)
    return (0, 0)

def net_force(aircraft):
    return vector_sum(engine_thrust_force(aircraft), wing_aerodynamic_force(aircraft), stabalizer_aerodynamic_force(aircraft), parasitic_drag_force(aircraft), weight_force(aircraft), rolling_resistance_force(aircraft))

def linear_acceleration(aircraft):  
    return scale_vector(net_force(aircraft), 1.0 / aircraft.mass)
    
def update_linear_velocity_and_position(aircraft, delta_time):
    # Use Euler method
    acceleration = linear_acceleration(aircraft)
    aircraft.linear_velocity = vector_sum(aircraft.linear_velocity, scale_vector(acceleration, delta_time))
    aircraft.position = vector_sum(aircraft.position, scale_vector(aircraft.linear_velocity, delta_time))

def compute_moment(force, angle_of_moment_arm_radians, distance_from_centre_of_rotation):
    tangential_force = component_of_vector_in_direction_radians(force, angle_of_moment_arm_radians + math.radians(90))
    return tangential_force * distance_from_centre_of_rotation

def wing_moment(aircraft):
    angle_of_moment_arm_radians = aircraft.pitch_angle_radians + aircraft.wing_angle_from_centre_of_gravity_radians
    return compute_moment(wing_aerodynamic_force(aircraft), angle_of_moment_arm_radians, aircraft.wing_distance_to_centre_of_gravity)

def stabilizer_moment(aircraft):
    angle_of_moment_arm_radians =  aircraft.pitch_angle_radians + aircraft.stabilizer_angle_from_centre_of_gravity_radians
    return compute_moment(stabalizer_aerodynamic_force(aircraft), angle_of_moment_arm_radians, aircraft.stabilizer_distance_centre_of_gravity)

def net_moment(aircraft):
    return wing_moment(aircraft) + stabilizer_moment(aircraft)

def angular_acceleration(aircraft):
    return net_moment(aircraft) / aircraft.plane_moment_of_inertia
    
def update_angular_velocity_and_pitch(aircraft, delta_time):
    acceleration = angular_acceleration(aircraft)                                    # radians per second per second
    aircraft.angular_velocity += acceleration * delta_time                           # radians per second
    aircraft.pitch_angle_radians += aircraft.angular_velocity * delta_time           # radians
    aircraft.pitch_angle_radians = normalize_angle(aircraft.pitch_angle_radians)     # ensure angle is between -pi and +pi

def correct_for_ground(aircraft):     
    if is_on_ground(aircraft):
        # don't fall below ground
        aircraft.position = (aircraft.position[0], 0)
        aircraft.linear_velocity = (aircraft.linear_velocity[0], 0)

        # don't roll backwards due to over zealous rolling resistance
        if aircraft.linear_velocity[0] < 0 :
            aircraft.linear_velocity = (0, 0)

        # front wheel prevents aircraft from pitching forward on the ground
        if aircraft.pitch_angle_radians < 0:
            aircraft.pitch_angle_radians = 0
            aircraft.angular_velocity = 0             
   
def simulate_one_time_step(aircraft, delta_time):   
    update_engine_rpm(aircraft, delta_time)
    update_angular_velocity_and_pitch(aircraft, delta_time)
    update_linear_velocity_and_position(aircraft, delta_time)
    correct_for_ground(aircraft)
    aircraft.time_elapsed += delta_time

def simulate_multiple_time_steps(aircraft, number_of_time_steps, delta_time):
    for step in range(number_of_time_steps):
        simulate_one_time_step(aircraft, delta_time)

In [ ]:
from unit_tests import test_correctness

test_correctness(vector_sum,advance_ratio,air_density_at,airspeed,angle_to_relative_airflow_radians,compute_moment,engine_angular_acceleration,engine_thrust_force,engine_torque_generated,
                 engine_torque_load,induced_drag_force,induced_drag_magnitude,is_on_ground,lift_force,magnitude_of_lift,net_engine_torque,net_force,parasitic_drag_force,
                 propeller_area,propeller_radians_per_second,propeller_rpm,propeller_torque_load,simulate_multiple_time_steps,stabalizer_aerodynamic_force,
                 stabilizer_moment,total_aerodynamic_force,update_angular_velocity_and_pitch,update_engine_rpm,update_linear_velocity_and_position,wing_aerodynamic_force,wing_moment,
                 air_density,angular_acceleration,linear_acceleration)

In [ ]:
from code_analyser import assess_part_b_and_c_of_assessment_criteria

assess_part_b_and_c_of_assessment_criteria(vector_sum,advance_ratio,airspeed,angle_to_relative_airflow_radians,compute_moment,engine_angular_acceleration,engine_thrust_force,engine_torque_generated,
                 engine_torque_load,induced_drag_force,induced_drag_magnitude,is_on_ground,lift_force,magnitude_of_lift,net_engine_torque,net_force,parasitic_drag_force,
                 propeller_area,propeller_radians_per_second,propeller_rpm,propeller_torque_load,simulate_multiple_time_steps,stabalizer_aerodynamic_force,
                 stabilizer_moment,total_aerodynamic_force,update_angular_velocity_and_pitch,update_engine_rpm,update_linear_velocity_and_position,wing_aerodynamic_force,wing_moment,
                 air_density,angular_acceleration,linear_acceleration)

In [ ]:
from tecnam import TecnamP92
from gui import simulate

%matplotlib widget

simulate(TecnamP92, simulate_multiple_time_steps, 30, 0.01)